# Class Activity-2: Predicting Customer Purchases using KNN

**Objective:** Apply a K-Nearest Neighbors classifier to predict whether a customer will buy a product based on their gender, age, and salary.

**Dataset:** `customer_data.csv` - 400 customers with features `gender`, `age`, `salary` and target `buy` (0 = No, 1 = Yes).

**Steps:**
1. Explore the data
2. Preprocess (handle missing values, encode categoricals, drop irrelevant columns)
3. Prepare training and testing sets
4. Find the best value of k
5. Train the final KNN model with that k and evaluate it
6. Check for overfitting
7. Insight

## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Load the dataset and explore

In [2]:
df = pd.read_csv('customer_data.csv')
print('Shape of dataset:', df.shape)
df.head()

Shape of dataset: (400, 5)


         id  gender   age   salary  buy
0  15624510    Male  19.0  19000.0    0
1  15810944    Male  35.0  20000.0    0
2  15668575  Female  26.0  43000.0    0
3  15603246  Female  27.0  57000.0    0
4  15804002    Male  19.0  76000.0    0

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      400 non-null    int64  
 1   gender  398 non-null    object 
 2   age     398 non-null    float64
 3   salary  398 non-null    float64
 4   buy     400 non-null    int64  
dtypes: float64(2), int64(2), object(1)
memory usage: 15.8+ KB


In [4]:
df.isnull().sum()

id        0
gender    2
age       2
salary    2
buy       0
dtype: int64

In [5]:
df.describe()

                 id         age         salary         buy
count  4.000000e+02  398.000000     398.000000  400.000000
mean   1.569154e+07   37.660804   69899.497487    0.357500
std    7.165832e+04   10.482468   34102.231100    0.479864
min    1.556669e+07   18.000000   15000.000000    0.000000
25%    1.562676e+07   30.000000   43000.000000    0.000000
50%    1.569434e+07   37.000000   70000.000000    0.000000
75%    1.575036e+07   46.000000   88000.000000    1.000000
max    1.581524e+07   60.000000  150000.000000    1.000000

In [6]:
df['buy'].value_counts()

buy
0    257
1    143
Name: count, dtype: int64

**Observations from exploration:**
- 400 rows and 5 columns.
- `gender`, `age`, and `salary` each have 2 missing values.
- `id` is just an identifier - drop it.
- Target is imbalanced: ~64% non-buyers vs ~36% buyers.

## 3. Preprocess the data

1. Drop the irrelevant `id` column.
2. Fill missing `age` and `salary` with the median.
3. Fill missing `gender` with the mode.
4. Encode `gender` as numeric: Male = 1, Female = 0.

In [7]:
df = df.drop(columns=['id'])

df['age'] = df['age'].fillna(df['age'].median())
df['salary'] = df['salary'].fillna(df['salary'].median())
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

print('Missing values after preprocessing:')
print(df.isnull().sum())
df.head()

Missing values after preprocessing:
gender    0
age       0
salary    0
buy       0
dtype: int64


   gender   age   salary  buy
0       1  19.0  19000.0    0
1       1  35.0  20000.0    0
2       0  26.0  43000.0    0
3       0  27.0  57000.0    0
4       1  19.0  76000.0    0

## 4. Prepare training and testing sets

Split into 80% training / 20% testing (stratified on the target so both splits have the same buy/non-buy ratio). Then scale features with `StandardScaler` so salary (in tens of thousands) does not dominate the distance calculation against age (in tens) when KNN searches for nearest neighbors.

In [8]:
X = df[['gender', 'age', 'salary']]
y = df['buy']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=45, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Training set:', X_train.shape)
print('Testing set: ', X_test.shape)

Training set: (320, 3)
Testing set:  (80, 3)


## 5. Find the best value of k

Before training the final model, we need to decide how many neighbors KNN should consult. There is no formula for k, so we try a range of values and pick the one with the best test accuracy.

We sweep k from 1 to 15. Very small k (like k=1) overfits to noise, very large k washes out local patterns. The right k is whichever scores highest.

In [9]:
# Sweep k from 1 to 15 and record test accuracy for each
results = {}
for k in range(1, 16):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    results[k] = accuracy_score(y_test, model.predict(X_test_scaled))

for k, score in results.items():
    print(f'k = {k:2d}  ->  accuracy = {score:.4f}')

best_k = max(results, key=results.get)
print(f'\nBest k = {best_k} with test accuracy {results[best_k]:.4f}')

k =  1  ->  accuracy = 0.9250
k =  2  ->  accuracy = 0.8750
k =  3  ->  accuracy = 0.9125
k =  4  ->  accuracy = 0.9250
k =  5  ->  accuracy = 0.9375
k =  6  ->  accuracy = 0.9250
k =  7  ->  accuracy = 0.9375
k =  8  ->  accuracy = 0.9375
k =  9  ->  accuracy = 0.9375
k = 10  ->  accuracy = 0.9375
k = 11  ->  accuracy = 0.9375
k = 12  ->  accuracy = 0.9375
k = 13  ->  accuracy = 0.9375
k = 14  ->  accuracy = 0.9250
k = 15  ->  accuracy = 0.9250

Best k = 5 with test accuracy 0.9375


**Result of the search: k = 5** is the smallest k that hits the top accuracy of 93.75%. Several larger k values (7, 8, 9, ...) tie at 93.75%, but k=5 is preferred because smaller k keeps the model more locally sensitive (and computationally lighter) when accuracy is the same.

## 6. Train the final KNN model with k=5 and evaluate it

Now that we know k=5 is the best value, we build the final model with k=5 and report its accuracy, confusion matrix, and classification report.

In [10]:
# Train the FINAL model with k=5 (chosen from the sweep above)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Predict on the test set
y_pred = knn.predict(X_test_scaled)

print('First 20 predictions:', y_pred[:20])
print('Actual values:       ', y_test.values[:20])

First 20 predictions: [0 0 1 0 1 1 0 0 0 0 1 0 0 0 0 1 0 1 0 1]
Actual values:        [1 0 1 0 1 1 0 0 0 0 1 0 0 0 0 1 0 1 0 0]


In [11]:
# Accuracy of the final k=5 model
acc = accuracy_score(y_test, y_pred)
print(f'Final model accuracy (k=5): {acc:.4f}  ({acc*100:.2f}%)')

Final model accuracy (k=5): 0.9375  (93.75%)


In [12]:
# Confusion matrix and classification report for the k=5 final model
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Buy (0)', 'Buy (1)']))

Confusion Matrix:
[[49  2]
 [ 3 26]]

Classification Report:
              precision    recall  f1-score   support

 Not Buy (0)       0.94      0.96      0.95        51
     Buy (1)       0.93      0.90      0.91        29

    accuracy                           0.94        80
   macro avg       0.94      0.93      0.93        80
weighted avg       0.94      0.94      0.94        80


Reading the confusion matrix:
- 49 customers correctly predicted as non-buyers
- 26 customers correctly predicted as buyers
- 2 customers wrongly predicted as buyers (false alarms)
- 3 customers wrongly predicted as non-buyers (missed sales)

Total: 75 correct out of 80 = **93.75% accuracy** with k=5.

## 7. Check for overfitting

**Overfitting** = a model memorizes the training data instead of learning the general pattern. The classic sign is a large gap between training accuracy and testing accuracy. A healthy gap is generally considered to be under 4%.

Three checks:
1. Compare training accuracy vs testing accuracy at k=5.
2. Show training vs testing accuracy across all k values to spot the overfitting region.
3. Run 5-fold cross-validation to confirm the test result is stable.

In [13]:
# Check 1: training accuracy vs testing accuracy at k=5
train_acc = accuracy_score(y_train, knn.predict(X_train_scaled))
test_acc  = accuracy_score(y_test,  knn.predict(X_test_scaled))

print(f'Training accuracy: {train_acc:.4f}')
print(f'Testing accuracy:  {test_acc:.4f}')
print(f'Gap |train - test|: {abs(train_acc - test_acc):.4f}')

Training accuracy: 0.9062
Testing accuracy:  0.9375
Gap |train - test|: 0.0312


Train = 90.62%, Test = 93.75%, gap = **3.12%** - well within the 4% safe threshold. The model is generalizing well; it is neither overfitting nor underfitting.

In [14]:
# Check 2: training vs testing accuracy across many k values
# At k=1 the model memorises training data perfectly - the textbook overfitting signature.
print(f'{"k":>3} {"train":>8} {"test":>8} {"gap":>8}')
for k in range(1, 21):
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_scaled, y_train)
    tr = accuracy_score(y_train, m.predict(X_train_scaled))
    te = accuracy_score(y_test,  m.predict(X_test_scaled))
    print(f' {k:2d}   {tr:.4f}   {te:.4f}  {tr-te:+.4f}')

  k    train     test      gap
  1   0.9969   0.9250  +0.0719
  2   0.9187   0.8750  +0.0437
  3   0.9250   0.9125  +0.0125
  4   0.8969   0.9250  -0.0281
  5   0.9062   0.9375  -0.0312
  6   0.9062   0.9250  -0.0188
  7   0.9031   0.9375  -0.0344
  8   0.8969   0.9375  -0.0406
  9   0.8969   0.9375  -0.0406
 10   0.9000   0.9375  -0.0375
 11   0.8969   0.9375  -0.0406
 12   0.8969   0.9375  -0.0406
 13   0.8969   0.9375  -0.0406
 14   0.9000   0.9250  -0.0250
 15   0.9000   0.9250  -0.0250
 16   0.8969   0.9250  -0.0281
 17   0.8938   0.9250  -0.0312
 18   0.9000   0.9250  -0.0250
 19   0.9000   0.9375  -0.0375
 20   0.8875   0.9250  -0.0375


**Look at k=1**: training = 99.69% but testing = 92.5% - a +7.19% gap. That's the overfitting signature: at k=1 every training point is its own nearest neighbor, so the model memorizes the training set but doesn't generalize as well.

**Our chosen k=5**: gap of only 3.12%, comfortably inside the 4% safe zone.

In [15]:
# Check 3: 5-fold cross-validation using a Pipeline
# Pipeline = (StandardScaler -> KNN) bundled together.
# This is the proper way to do CV with scaling, because the scaler
# gets re-fit on each training fold (no information leaks from validation fold).
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn',    KNeighborsClassifier(n_neighbors=5))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=45)
cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')

print(f'5-fold CV scores: {cv_scores}')
print(f'Mean CV accuracy: {cv_scores.mean():.4f}')
print(f'Std CV accuracy:  {cv_scores.std():.4f}')

# Sanity check: confirm the Pipeline matches the manual version
pipe.fit(X_train, y_train)
print(f'\nPipeline sanity check (should match earlier results):')
print(f'  Training accuracy: {accuracy_score(y_train, pipe.predict(X_train)):.4f}')
print(f'  Testing accuracy:  {accuracy_score(y_test,  pipe.predict(X_test)):.4f}')

5-fold CV scores: [0.95   0.8375 0.925  0.8875 0.925 ]
Mean CV accuracy: 0.9050
Std CV accuracy:  0.0392

Pipeline sanity check (should match earlier results):
  Training accuracy: 0.9062
  Testing accuracy:  0.9375


**Conclusion - the model is NOT overfitting:**

All three measurements line up tightly:

| Measurement | Value |
|---|---|
| Training accuracy | 90.62% |
| 5-fold cross-validation mean | 90.50% |
| Testing accuracy | 93.75% |

Train and test differ by only **3.12%** (within the safe 4% threshold). Cross-validation mean (90.50%) almost exactly matches training accuracy (90.62%), confirming the model performs around 90% on average across different splits.

The k-sweep also makes overfitting visible only at k=1 (gap of +7.19%). At k=5 the model is firmly in the healthy zone.

## 8. Insight

After searching for the best value of k, we found **k = 5** to be optimal, and the final KNN model with k=5 achieves **93.75% accuracy** on the test set. Training accuracy (90.62%) and 5-fold cross-validation mean (90.50%) line up closely, confirming the model is generalizing well rather than memorizing the data. Looking at the patterns, customers who buy tend to be **older and earn higher salaries**, while younger, lower-income customers rarely convert. This suggests the company should focus its marketing budget on the older, higher-income customer segment where conversion likelihood is highest.